# Spatial evaluation of yield variation risk of major grains under climate change

**Fixed-technology scenario and ML-based climate hotspot analysis**

- **Objective**: Under a *fixed technology* assumption (no adaptation), evaluate yield variation risk by space (Area) using ML-predicted yields for future climate (e.g. 2100).
- **Data**: Historical regression data (e.g. Year≤2014), future climate scenarios (SSP245, SSP370).
- **Outputs**: Risk metrics by Area (e.g. yield change, CV), identification of *climate hotspots* (high-risk areas).
- Optional: Load the reference document (.docx) in the first code cell to align with the paper.

## 1. Optional: Load reference document (.docx)

Set `DOCX_PATH` to your file path and run to extract and display the document text.

In [1]:
import zipfile
import re
import os

DOCX_PATH = r'C:\Users\xyz19\Downloads\_気候変動下における主要穀物の収量変動リスクの空間的評価 —— 技術固定シナリオと機械学習による気候ホットスポットの分析 (1) (1).docx'

def extract_docx_text(path):
    if not os.path.isfile(path):
        print('File not found. Set DOCX_PATH to your .docx path.')
        return None
    with zipfile.ZipFile(path, 'r') as z:
        xml = z.read('word/document.xml').decode('utf-8')
    text = re.sub(r'<w:t[^>]*>([^<]*)</w:t>', r'\1', xml)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

doc_text = extract_docx_text(DOCX_PATH)
if doc_text:
    print('--- Document text (first 4000 chars) ---')
    print(doc_text[:4000])
    print('...' if len(doc_text) > 4000 else '')

--- Document text (first 4000 chars) ---
1. 緒言 (Introduction) 現在、気候変動シナリオSSP2-4.5およびSSP3-7.0下における世界の食糧需給は、これまでに考えられない程度で変化していくことが予測される。既存の物理モデルはポテンシャル評価に留まる傾向がある。そこで本研究では、機械学習（XGBoost）と農学的な「実効供給補正係数（$\alpha$）」を統合することで、「現実的な供給能力」という一貫した基準を用いて世界のリスクを再評価した 。 本アプローチにより、物理的な気候ストレスが、農学的な地力制約や社会的な適応の硬直性と結びつき、どれほど収量への影響するのかを定量化する。その結果に基づき、将来の食糧安全保障を担保するための具体的な農業インフラ政策を提言することを目的とする。 2. 手法 (Methodology) 2.1 データセットの構築と空間統合 本研究では、 目的変数として 1961年から2013年までの各国別・作物別（米、トウモロコシ、小麦）の単位面積当たりの収量データ（FAOSTAT）を 採用した。 また、説明変数として、World Bank GroupのClimate Change Knowledge Portalにある気候データである、 ACCESS-CM2モデルの 、最高気温(TXX)、生育期間（Growing Season Length: GSL）、および水分不足の指標となるSPEI（Standardised Precipitation-Evapotranspiration Index）、相対湿度(Hurs）を採用した。加えて、技術進歩を考慮するために yearを技術変数 を表現するものとして採用する。そして、この目的変数と説明変数のそれぞれの国をマッチングさせ、成功した156ヶ国でモデルを構築した。 将来予測における気候データについては、単年度の極端な気象ノイズが予測結果を不安定化させるのを防ぐため、時間的な平滑化処理を行った。具体的には、ターゲットイヤーである2100年の気候入力値として、直前の2096年から2099年までの4カ年平均値を採用した。これにより、短期的な自然変動の影響を緩和し、21世紀末における「気候の平均状態」に対する収量応答を評価する設計とした。 2.2 超

## 2. Setup and load regression data

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import r2_score
import xgboost as xgb

base_dir = r'C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット'
os.chdir(base_dir)

regressionDF = pd.read_excel(os.path.join(base_dir, 'regression_df_20260112_233447.xlsx'))
print('Regression data:', regressionDF.shape)
print('Year range:', regressionDF['Year'].min(), '-', regressionDF['Year'].max())

Regression data: (16388, 8)
Year range: 1961 - 2013


## 3. Feature build and train XGBoost (fixed technology)

In [3]:
climate_features = ['SPEI', 'GSL', 'Hurs', 'TXX']
area_dummies = pd.get_dummies(regressionDF['Area'], prefix='Area')
item_dummies = pd.get_dummies(regressionDF['Item'], prefix='Item')
year_feature = regressionDF[['Year']].copy()

X = pd.concat([
    regressionDF[climate_features],
    area_dummies,
    item_dummies,
    year_feature
], axis=1)
y = regressionDF['Yield'].copy()

train_mask = regressionDF['Year'] <= 2014
X_train = X.loc[train_mask].astype(np.float64)
y_train = y.loc[train_mask]

train_areas = sorted(regressionDF.loc[train_mask, 'Area'].unique().tolist())
train_items = sorted(regressionDF.loc[train_mask, 'Item'].unique().tolist())
feature_cols = X_train.columns.tolist()

best_params = {
    'colsample_bytree': 1.0,
    'learning_rate': 0.2,
    'max_depth': 5,
    'n_estimators': 300,
    'subsample': 0.9,
    'random_state': 42,
    'n_jobs': -1
}

model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train)
r2 = r2_score(y_train, model.predict(X_train))
print('Train R2:', round(r2, 4))
print('Areas:', len(train_areas), '| Items:', len(train_items), '| X cols:', X.shape[1])

Train R2: 0.9624
Areas: 156 | Items: 3 | X cols: 164


## 4. Build future X (2100) and predict for each scenario

In [4]:
def build_X_future(future_df, train_areas, train_items, feature_cols, climate_features):
    areas_to_use = [a for a in train_areas if a in set(future_df['Area'].unique())]
    rows = []
    for area in areas_to_use:
        r = future_df[future_df['Area'] == area].iloc[0]
        for item in train_items:
            rows.append({
                'Area': area, 'Item': item, 'Year': 2100,
                'SPEI': r['SPEI'], 'GSL': r['GSL'], 'Hurs': r['Hurs'], 'TXX': r['TXX']
            })
    if not rows:
        return pd.DataFrame(columns=feature_cols), pd.DataFrame(columns=['Area', 'Item'])
    df = pd.DataFrame(rows)
    area_cat = pd.Categorical(df['Area'], categories=train_areas)
    item_cat = pd.Categorical(df['Item'], categories=train_items)
    area_dum = pd.get_dummies(area_cat, prefix='Area')
    item_dum = pd.get_dummies(item_cat, prefix='Item')
    X_f = pd.concat([df[climate_features], area_dum, item_dum, df[['Year']]], axis=1)
    for c in feature_cols:
        if c not in X_f.columns:
            X_f[c] = 0
    X_f = X_f[feature_cols]
    meta = df[['Area', 'Item']].copy()
    return X_f, meta

# Load future climate CSVs (same as predict_2100_bootstrap)
future_245 = pd.read_csv(os.path.join(base_dir, 'future_2100_features_SSP245_nfuture_2096-2099_mean_20260129_043901.csv'))
future_370 = pd.read_csv(os.path.join(base_dir, 'future_2100_features_SSP370_bfuture_2096-2099_mean_20260129_043854.csv'))

X_245, meta_245 = build_X_future(future_245, train_areas, train_items, feature_cols, climate_features)
X_370, meta_370 = build_X_future(future_370, train_areas, train_items, feature_cols, climate_features)

y_245 = model.predict(X_245)
y_370 = model.predict(X_370)

pred_245 = meta_245.copy()
pred_245['yield_2100'] = y_245
pred_245['scenario'] = 'SSP245'
pred_370 = meta_370.copy()
pred_370['yield_2100'] = y_370
pred_370['scenario'] = 'SSP370'

print('SSP245 predictions:', len(pred_245))
print('SSP370 predictions:', len(pred_370))

SSP245 predictions: 468
SSP370 predictions: 468


## 5. Baseline (historical) yield by Area–Item for comparison

In [5]:
# Use recent period as baseline (e.g. 2011-2013 mean)
baseline_mask = (regressionDF['Year'] >= 2011) & (regressionDF['Year'] <= 2013)
baseline = regressionDF.loc[baseline_mask].groupby(['Area', 'Item'])['Yield'].mean().reset_index()
baseline = baseline.rename(columns={'Yield': 'yield_baseline'})
print('Baseline (2011-2013 mean) rows:', len(baseline))

Baseline (2011-2013 mean) rows: 350


## 6. Risk metrics by Area (yield change, variation, hotspot flag)

In [6]:
def risk_by_area(pred_df, baseline_df, scenario_name):
    m = pred_df.merge(baseline_df, on=['Area', 'Item'])
    m['yield_change_pct'] = (m['yield_2100'] - m['yield_baseline']) / m['yield_baseline'].replace(0, np.nan) * 100
    by_area = m.groupby('Area').agg(
        yield_baseline=('yield_baseline', 'mean'),
        yield_2100=('yield_2100', 'mean'),
        yield_change_pct=('yield_change_pct', 'mean'),
        n_items=('Item', 'count')
    ).reset_index()
    by_area['scenario'] = scenario_name
    return by_area

risk_245 = risk_by_area(pred_245, baseline, 'SSP245')
risk_370 = risk_by_area(pred_370, baseline, 'SSP370')

# Combine and define hotspot: e.g. yield decline > 10% in either scenario
risk_245['hotspot_245'] = risk_245['yield_change_pct'] < -10
risk_370['hotspot_370'] = risk_370['yield_change_pct'] < -10

hotspot_areas_245 = set(risk_245.loc[risk_245['hotspot_245'], 'Area'])
hotspot_areas_370 = set(risk_370.loc[risk_370['hotspot_370'], 'Area'])
hotspot_either = hotspot_areas_245 | hotspot_areas_370

print('Areas with yield decline > 10% (SSP245):', len(hotspot_areas_245))
print('Areas with yield decline > 10% (SSP370):', len(hotspot_areas_370))
print('Areas hotspot in either scenario:', len(hotspot_either))

Areas with yield decline > 10% (SSP245): 23
Areas with yield decline > 10% (SSP370): 35
Areas hotspot in either scenario: 35


## 7. Summary table and hotspot list

In [7]:
summary = risk_245[['Area', 'yield_change_pct']].rename(columns={'yield_change_pct': 'change_SSP245'})
summary = summary.merge(
    risk_370[['Area', 'yield_change_pct']].rename(columns={'yield_change_pct': 'change_SSP370'}),
    on='Area'
)
summary['hotspot'] = summary['Area'].isin(hotspot_either)
summary = summary.sort_values('change_SSP370')
display(summary.head(20))
print('--- Hotspot countries (either scenario) ---')
print(sorted(hotspot_either))

,Area,change_SSP245,change_SSP370,hotspot
69,Ireland,-46.925968,-38.937805,True
130,South Africa,-37.517180,-34.533075,True
89,Maldives,-33.592899,-33.592899,True
94,Mexico,-33.052721,-31.278719,True
113,Peru,-29.770229,-29.324642,True
80,Lebanon,-20.020650,-28.344125,True
14,Belize,-26.148907,-25.604305,True
115,Poland,-6.674665,-25.402666,True
149,United Arab Emirates,-45.521587,-23.739807,True
57,Greece,-3.034638,-23.732516,True


--- Hotspot countries (either scenario) ---
['Albania', 'Austria', 'Belarus', 'Belize', 'Brazil', 'Cambodia', 'Canada', 'Chile', 'Cyprus', 'Ethiopia', 'Germany', 'Greece', 'Indonesia', 'Ireland', 'Kenya', "Lao People's Democratic Republic", 'Lebanon', 'Lesotho', 'Luxembourg', 'Maldives', 'Malta', 'Mexico', 'New Zealand', 'Peru', 'Philippines', 'Poland', 'Saint Vincent and the Grenadines', 'Slovenia', 'South Africa', 'Spain', 'Sri Lanka', 'Switzerland', 'Türkiye', 'United Arab Emirates', 'United States of America']


## 8. Export results (optional)

In [8]:
out_risk = pd.concat([risk_245, risk_370], ignore_index=True)
out_path = os.path.join(base_dir, 'climate_yield_risk_by_area.csv')
out_risk.to_csv(out_path, index=False)
print('Saved:', out_path)

summary.to_csv(os.path.join(base_dir, 'climate_hotspots_summary.csv'), index=False)
print('Saved: climate_hotspots_summary.csv')

Saved: C:\Users\xyz19\OneDrive\デスクトップ\予測モデルデータセット\climate_yield_risk_by_area.csv
Saved: climate_hotspots_summary.csv
